Cell 1 — Mount Google Drive

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# Experiment M2 - MMS adapter fine-tuning on Corpus V1.1
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Cell 2 — Define project paths

In [ ]:
# ============================================================
# CELL 2: Define M2 project paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

TRAIN_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "train.csv"
)

VALIDATION_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "validation.csv"
)

M2_TOKENIZER_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_M2_tokenizer_v1_1"
)

M2_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_M2_corpus_v1_1"
)

M2_OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "mms_M2_adapter_v1_1"
)

print("Project:", PROJECT_ROOT.exists())
print("Train:", TRAIN_CSV.exists())
print("Validation:", VALIDATION_CSV.exists())

Project: True
Train: True
Validation: True


Cell 3 — Install dependencies

In [ ]:
# ============================================================
# CELL 3: Install MMS adapter dependencies
# ============================================================

!pip install -q transformers datasets accelerate jiwer soundfile safetensors

Cell 4 — Load Corpus V1.1

In [ ]:
# ============================================================
# CELL 4: Load Corpus V1.1
# ============================================================

import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.read_csv(TRAIN_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)

dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_df,
        preserve_index=False
    )
})

print(dataset)

print(
    "\nTrain duration:",
    round(train_df["duration_seconds"].sum() / 3600, 3),
    "hours"
)

print(
    "Validation duration:",
    round(validation_df["duration_seconds"].sum() / 60, 2),
    "minutes"
)

DatasetDict({
    train: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 133
    })
})

Train duration: 4.638 hours
Validation duration: 18.12 minutes


Cell 5 — Build the Corpus V1.1 character vocabulary

In [ ]:
# ============================================================
# CELL 5: Build Corpus V1.1 CTC vocabulary
# ============================================================

all_text = (
    train_df["transcription"].astype(str).tolist()
    + validation_df["transcription"].astype(str).tolist()
)

characters = sorted(
    set("".join(all_text))
)

print("Corpus characters:")
print(characters)

print("\nNumber of characters:", len(characters))
print("ř present:", "ř" in characters)

Corpus characters:
[' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'u', 'w', 'x', 'y', 'z', 'ǧ', 'ɛ', 'ɣ', 'ʷ', 'ḍ', 'ḥ', 'ṣ', 'ṭ', 'ẓ']

Number of characters: 34
ř present: False


Cell 6 — Build the MMS nested adapter vocabulary

In [ ]:
# ============================================================
# CELL 6: Build nested MMS adapter vocabulary
# ============================================================

import json

TARGET_LANG = "rif"

vocab_dict = {
    char: idx
    for idx, char in enumerate(characters)
}

# Replace normal space with visible CTC delimiter
space_id = vocab_dict.pop(" ")
vocab_dict["|"] = space_id

# Add CTC special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

# MMS adapters expect language-keyed vocabulary
nested_vocab = {
    TARGET_LANG: vocab_dict
}

print("Target language:", TARGET_LANG)
print("Vocabulary size:", len(vocab_dict))
print("ř present:", "ř" in vocab_dict)

Target language: rif
Vocabulary size: 36
ř present: False


Cell 7 — Save the M2 vocabulary

In [ ]:
# ============================================================
# CELL 7: Save M2 MMS adapter vocabulary
# ============================================================

M2_TOKENIZER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VOCAB_PATH = (
    M2_TOKENIZER_DIR
    / "vocab.json"
)

with open(
    VOCAB_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        nested_vocab,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:")
print(VOCAB_PATH)

Saved:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_M2_tokenizer_v1_1/vocab.json


Cell 8 — Create the M2 tokenizer

In [ ]:
# ============================================================
# CELL 8: Create MMS adapter tokenizer
# ============================================================

from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    target_lang=TARGET_LANG
)

print("Target language:", TARGET_LANG)
print("Tokenizer size:", len(tokenizer))
print("ř in tokenizer:", "ř" in tokenizer.get_vocab())

Target language: rif
Tokenizer size: 38
ř in tokenizer: False


Cell 9 — Create feature extractor and processor

In [ ]:
# ============================================================
# CELL 9: Create MMS processor
# ============================================================

from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

print("Processor ready.")

Processor ready.


Cell 10 — Verify tokenizer round-trip

In [ ]:
# ============================================================
# CELL 10: Verify M2 tokenizer
# ============================================================

sample_text = train_df.iloc[0]["transcription"]

encoded = tokenizer(sample_text)

decoded = tokenizer.decode(
    encoded.input_ids,
    group_tokens=False
)

print("Original:")
print(sample_text)

print("\nDecoded:")
print(decoded)

print("\nExact match:", sample_text == decoded)

Original:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Decoded:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Exact match: True


Cell 11 — Define audio preprocessing

In [ ]:
# ============================================================
# CELL 11: Define M2 audio/text preprocessing
# ============================================================

import soundfile as sf


def prepare_m2_dataset(example):

    audio_file = (
        PROJECT_ROOT
        / example["audio_path"]
    )

    audio, sampling_rate = sf.read(
        audio_file
    )

    inputs = processor(
        audio,
        sampling_rate=sampling_rate
    )

    example["input_values"] = (
        inputs.input_values[0]
    )

    example["input_length"] = len(
        example["input_values"]
    )

    example["labels"] = tokenizer(
        example["transcription"]
    ).input_ids

    return example


print("M2 preprocessing ready.")

M2 preprocessing ready.


Cell 12 — Test preprocessing on one segment

In [ ]:
# ============================================================
# CELL 12: Test M2 preprocessing
# ============================================================

sample = prepare_m2_dataset(
    dataset["train"][0]
)

print("Input samples:", len(sample["input_values"]))

print("\nReference:")
print(sample["transcription"])

print("\nDecoded labels:")
print(
    tokenizer.decode(
        sample["labels"],
        group_tokens=False
    )
)

Input samples: 60160

Reference:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Decoded labels:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta


Cell 13 — Preprocess the full dataset

In [ ]:
# ============================================================
# CELL 13: Preprocess full Corpus V1.1 for MMS M2
# ============================================================

m2_dataset = dataset.map(
    prepare_m2_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1
)

print(m2_dataset)

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 133
    })
})


Cell 14 — Save the processed M2 dataset

In [ ]:
# ============================================================
# CELL 14: Save preprocessed MMS M2 dataset
# ============================================================

m2_dataset.save_to_disk(
    str(M2_DATASET_PATH)
)

print("Saved to:")
print(M2_DATASET_PATH)

Saving the dataset (0/3 shards):   0%|          | 0/1472 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/133 [00:00<?, ? examples/s]

Saved to:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_M2_corpus_v1_1


Cell 15 — Define CTC collator

In [ ]:
# ============================================================
# CELL 15: Define MMS M2 CTC collator
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch


@dataclass
class DataCollatorCTCWithPadding:

    processor: Any
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[
            Dict[
                str,
                Union[List[int], torch.Tensor]
            ]
        ]
    ):

        input_features = [
            {
                "input_values":
                feature["input_values"]
            }
            for feature in features
        ]

        label_features = [
            {
                "input_ids":
                feature["labels"]
            }
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = (
            self.processor.tokenizer.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt"
            )
        )

        labels = (
            labels_batch["input_ids"]
            .masked_fill(
                labels_batch.attention_mask.ne(1),
                -100
            )
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor
)

print("M2 CTC collator ready.")

M2 CTC collator ready.


Cell 16 — Define WER/CER

In [ ]:
# ============================================================
# CELL 16: Define M2 WER and CER
# ============================================================

import numpy as np
from jiwer import wer, cer


def compute_metrics(pred):

    pred_ids = np.argmax(
        pred.predictions,
        axis=-1
    )

    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    pred_str = processor.batch_decode(
        pred_ids
    )

    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    return {
        "wer": wer(
            label_str,
            pred_str
        ) * 100,

        "cer": cer(
            label_str,
            pred_str
        ) * 100
    }


print("Metrics ready.")

Metrics ready.


Cell 17 — Load MMS-1B-all with Corpus V1.1 output layer

In [ ]:
# ============================================================
# CELL 17: Load MMS-1B-all for adapter fine-tuning
# ============================================================

from transformers import Wav2Vec2ForCTC

BASE_MODEL = "facebook/mms-1b-all"

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,

    vocab_size=len(tokenizer),

    pad_token_id=tokenizer.pad_token_id,

    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,

    ctc_loss_reduction="mean",

    ignore_mismatched_sizes=True
)

print("Base model:", BASE_MODEL)
print("Model vocabulary:", model.config.vocab_size)
print("Tokenizer vocabulary:", len(tokenizer))

print(
    "Parameters:",
    round(model.num_parameters() / 1e6, 1),
    "M"
)

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([38])            
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([38, 1280])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Base model: facebook/mms-1b-all
Model vocabulary: 38
Tokenizer vocabulary: 38
Parameters: 964.7 M


Cell 18 — Initialize MMS adapter layers

In [ ]:
# ============================================================
# CELL 18: Initialize MMS language adapters
# ============================================================

model.init_adapter_layers()

print("MMS adapter layers initialized.")

MMS adapter layers initialized.


Cell 19 — Freeze base model and activate adapters

In [ ]:
# ============================================================
# CELL 19: Freeze MMS base and train adapter weights
# ============================================================

# Freeze the huge MMS base network
model.freeze_base_model()

# Retrieve adapter-specific parameters
adapter_weights = model._get_adapters()

# Make adapter parameters trainable
for param in adapter_weights.values():
    param.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Trainable parameters:",
    round(trainable_params / 1e6, 2),
    "M"
)

print(
    "Total parameters:",
    round(total_params / 1e6, 2),
    "M"
)

print(
    "Trainable percentage:",
    round(
        100 * trainable_params / total_params,
        4
    ),
    "%"
)

Trainable parameters: 2.2 M
Total parameters: 964.7 M
Trainable percentage: 0.228 %


Cell 20 — Check GPU

In [ ]:
# ============================================================
# CELL 20: Check GPU before M2 training
# ============================================================

import torch

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    free, total = torch.cuda.mem_get_info()

    print(
        "Total GPU memory:",
        round(total / 1024**3, 2),
        "GB"
    )

    print(
        "Free GPU memory:",
        round(free / 1024**3, 2),
        "GB"
    )

CUDA available: True
GPU: Tesla T4
Total GPU memory: 14.56 GB
Free GPU memory: 14.46 GB


Cell 21 — Configure MMS M2 adapter training

In [ ]:
# ============================================================
# CELL 21: Configure MMS M2 adapter fine-tuning
# ============================================================

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(M2_OUTPUT_DIR),

    # T4-safe batching
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,

    # Adapter fine-tuning
    learning_rate=1e-3,
    num_train_epochs=10,
    warmup_steps=20,

    # GPU
    fp16=True,

    # Evaluation
    eval_strategy="epoch",

    # Checkpointing
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Logging
    logging_strategy="steps",
    logging_steps=25,

    report_to="none",
    seed=42,
)

print("M2 training arguments ready.")

M2 training arguments ready.


Cell 22 — Create the M2 Trainer

In [ ]:
# ============================================================
# CELL 22: Create MMS M2 Trainer
# ============================================================

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=m2_dataset["train"],
    eval_dataset=m2_dataset["validation"],

    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("M2 Trainer ready.")

M2 Trainer ready.


Cell 23 — Final sanity check

In [ ]:
# ============================================================
# CELL 23: Final sanity check before M2 training
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Train examples:", len(m2_dataset["train"]))
print("Validation examples:", len(m2_dataset["validation"]))

print("Tokenizer size:", len(tokenizer))
print("Model vocabulary size:", model.config.vocab_size)

print(
    "Trainable parameters:",
    round(trainable_params / 1e6, 2),
    "M"
)

print("Learning rate:", training_args.learning_rate)
print("Epochs:", training_args.num_train_epochs)

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Train examples: 1472
Validation examples: 133
Tokenizer size: 38
Model vocabulary size: 38
Trainable parameters: 2.2 M
Learning rate: 0.001
Epochs: 10
CUDA: True
GPU: Tesla T4


CELL 24: Start MMS M2 adapter fine-tuning

Cell 25 — Check CTC alignment feasibility

In [ ]:
# ============================================================
# CELL 25: Check CTC alignment feasibility
# ============================================================

def minimum_ctc_steps(labels):
    """
    Minimum number of CTC time steps required for a target sequence.
    Repeated adjacent labels require an intervening blank.
    """
    labels = list(labels)

    if len(labels) == 0:
        return 0

    repeated_adjacent = sum(
        labels[i] == labels[i - 1]
        for i in range(1, len(labels))
    )

    return len(labels) + repeated_adjacent


def check_ctc_feasibility(split_dataset, split_name):

    problems = []

    for i, example in enumerate(split_dataset):

        input_length = example["input_length"]
        labels = example["labels"]

        # Number of time steps produced by MMS after feature extraction
        output_length = int(
            model._get_feat_extract_output_lengths(
                input_length
            )
        )

        target_length = len(labels)
        min_required = minimum_ctc_steps(labels)

        if min_required > output_length:
            problems.append({
                "index": i,
                "split": split_name,
                "input_samples": input_length,
                "output_steps": output_length,
                "target_tokens": target_length,
                "min_ctc_steps": min_required,
                "shortfall": min_required - output_length,
            })

    return problems


train_ctc_problems = check_ctc_feasibility(
    m2_dataset["train"],
    "train"
)

validation_ctc_problems = check_ctc_feasibility(
    m2_dataset["validation"],
    "validation"
)

print("Train CTC problems:", len(train_ctc_problems))
print("Validation CTC problems:", len(validation_ctc_problems))

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CHECK 1: Show model/checkpoint folder sizes
# ============================================================

import os
from pathlib import Path

MODELS_DIR = PROJECT_ROOT / "models"

def folder_size_gb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = Path(root) / f
            try:
                total += fp.stat().st_size
            except:
                pass
    return total / (1024**3)

for p in sorted(MODELS_DIR.iterdir()):
    if p.is_dir():
        print(
            f"{p.name:40s}",
            f"{folder_size_gb(p):.2f} GB"
        )

mms_M2_adapter_v1_1                      7.22 GB
mms_tarifit_finetuned_v1_1               0.00 GB
whisper_small_corpus_v1                  5.40 GB
xlsr_300m_X2_corpus_v1_1                 6.99 GB
xlsr_300m_corpus_v1                      6.99 GB


In [ ]:
# ============================================================
# CHECK 2: Show MMS checkpoint sizes
# ============================================================

for experiment in [
    "mms_tarifit_finetuned_v1_1",
    "mms_M2_adapter_v1_1",
]:

    exp_dir = MODELS_DIR / experiment

    print("\n", experiment)

    if not exp_dir.exists():
        print("Not found")
        continue

    for ckpt in sorted(exp_dir.glob("checkpoint-*")):
        print(
            ckpt.name,
            f"{folder_size_gb(ckpt):.2f} GB"
        )


 mms_tarifit_finetuned_v1_1
checkpoint-184 0.00 GB
checkpoint-368 0.00 GB
checkpoint-552 0.00 GB

 mms_M2_adapter_v1_1
checkpoint-184 0.02 GB
checkpoint-368 3.61 GB
checkpoint-552 3.59 GB


In [ ]:
# ============================================================
# CELL 25B: Fast CTC feasibility check
# ============================================================

import numpy as np


def minimum_ctc_steps(labels):
    labels = np.asarray(labels)

    if len(labels) == 0:
        return 0

    repeats = np.sum(
        labels[1:] == labels[:-1]
    )

    return len(labels) + int(repeats)


def fast_ctc_check(split_dataset, split_name):

    input_lengths = split_dataset["input_length"]
    all_labels = split_dataset["labels"]

    # Vectorized MMS output lengths
    input_lengths_tensor = torch.tensor(
        input_lengths,
        dtype=torch.long
    )

    output_lengths = (
        model._get_feat_extract_output_lengths(
            input_lengths_tensor
        )
        .cpu()
        .numpy()
    )

    problems = []

    for idx, (out_len, labels) in enumerate(
        zip(output_lengths, all_labels)
    ):

        minimum = minimum_ctc_steps(labels)

        if minimum > out_len:
            problems.append({
                "index": idx,
                "split": split_name,
                "output_steps": int(out_len),
                "target_tokens": len(labels),
                "minimum_ctc_steps": minimum,
                "shortfall": minimum - int(out_len),
            })

    return problems


train_ctc_problems = fast_ctc_check(
    m2_dataset["train"],
    "train"
)

validation_ctc_problems = fast_ctc_check(
    m2_dataset["validation"],
    "validation"
)

print(
    "Train CTC problems:",
    len(train_ctc_problems)
)

print(
    "Validation CTC problems:",
    len(validation_ctc_problems)
)


Train CTC problems: 0
Validation CTC problems: 5


In [ ]:
# ============================================================
# CLEANUP: Remove failed M2 checkpoints to free Drive space
# ============================================================

import shutil

for ckpt_name in [
    "checkpoint-368",
    "checkpoint-552",
]:
    ckpt_path = (
        PROJECT_ROOT
        / "models"
        / "mms_M2_adapter_v1_1"
        / ckpt_name
    )

    if ckpt_path.exists():
        print("Deleting:", ckpt_path)
        shutil.rmtree(ckpt_path)

print("Cleanup finished.")

Deleting: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_M2_adapter_v1_1/checkpoint-368
Deleting: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_M2_adapter_v1_1/checkpoint-552
Cleanup finished.


Cell 26 — Show the problematic validation segments

In [ ]:
# ============================================================
# CELL 26: Inspect the 5 CTC-infeasible validation segments
# ============================================================

for problem in validation_ctc_problems:

    idx = problem["index"]
    row = validation_df.iloc[idx]

    print("=" * 80)
    print("Index:", idx)
    print("Segment:", row["segment_id"])
    print("Speaker:", row["speaker_group_id"])
    print("Duration:", round(row["duration_seconds"], 2), "seconds")

    print("Output steps:", problem["output_steps"])
    print("Target tokens:", problem["target_tokens"])
    print("Minimum CTC steps:", problem["minimum_ctc_steps"])
    print("Shortfall:", problem["shortfall"])

    print("\nTRANSCRIPTION:")
    print(row["transcription"])
    print()

Index: 111
Segment: REC138_SEG0025
Speaker: SPK010
Duration: 2.0 seconds
Output steps: 99
Target tokens: 151
Minimum CTC steps: 161
Shortfall: 62

TRANSCRIPTION:
walakin tin nnedni ɛejzent idan fekkarnt nitenti biannahu xminni ɣa xmi dd ɣa yas ad yaf nican nitenti ad xasent yazzer ad ijj illa tenni ikemmren awar

Index: 118
Segment: REC138_SEG0033
Speaker: SPK010
Duration: 6.0 seconds
Output steps: 299
Target tokens: 495
Minimum CTC steps: 536
Shortfall: 237

TRANSCRIPTION:
ttrass ttɛic ijj n lḥayat cwat teqseḥ lmuhimm igga as ɛuquba aḥenjir nni ittsemma imɣar dd tuɣa ddcar nni awmi tenni umi tewcin t imɣar dd imɣar ila axirih ibda ittirar ag iqqrinen nnes nnan as a weddi aqa cek qa yinni i ɣa ttɛiced aqa war llint ca d yemmak d babak lmuhimm kur marra iqqar as i tenni t yarbban iqqar as ixeṣṣ ad ayi tinid manwen illan d baba aqa qaɛ iḥramen i ked ttirarɣ qqarn ayi aqa wenni yinni war illi ca war illi ca d yemmak d babak u labudda li anna bnadem yarzzu x l aṣl

Index: 126
Segment: REC1

Cell 27 — Check the training problems too

In [ ]:
# ============================================================
# CELL 27: Verify real WAV durations for CTC-problem segments
# ============================================================

import soundfile as sf

for problem in validation_ctc_problems:

    idx = problem["index"]
    row = validation_df.iloc[idx]

    audio_file = PROJECT_ROOT / row["audio_path"]

    info = sf.info(audio_file)
    real_duration = info.frames / info.samplerate

    print("=" * 80)
    print("Segment:", row["segment_id"])
    print("Speaker:", row["speaker_group_id"])

    print("CSV duration:", row["duration_seconds"], "sec")
    print("Actual WAV duration:", round(real_duration, 3), "sec")

    print("Target chars:", len(row["transcription"]))

    print("\nAudio file:")
    print(audio_file)

    print()

Segment: REC138_SEG0025
Speaker: SPK010
CSV duration: 2.0 sec
Actual WAV duration: 2.0 sec
Target chars: 151

Audio file:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC138/REC138_SEG0025.wav

Segment: REC138_SEG0033
Speaker: SPK010
CSV duration: 6.0 sec
Actual WAV duration: 6.0 sec
Target chars: 495

Audio file:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC138/REC138_SEG0033.wav

Segment: REC138_SEG0041
Speaker: SPK010
CSV duration: 4.0 sec
Actual WAV duration: 4.0 sec
Target chars: 261

Audio file:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC138/REC138_SEG0041.wav

Segment: REC138_SEG0042
Speaker: SPK010
CSV duration: 2.0 sec
Actual WAV duration: 2.0 sec
Target chars: 92

Audio file:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC138/REC138_SEG0042.wav

Segment: REC138_SEG0045
Speaker: SPK010
CSV duration: 1.0 sec
Act

Cell 28 — Check for suspicious transcript density

In [ ]:
# ============================================================
# CELL 28: Remove CTC-infeasible validation pairs
# ============================================================

bad_indices = {
    problem["index"]
    for problem in validation_ctc_problems
}

valid_indices = [
    i
    for i in range(len(m2_dataset["validation"]))
    if i not in bad_indices
]

m2_validation_clean = (
    m2_dataset["validation"]
    .select(valid_indices)
)

validation_df_clean = (
    validation_df
    .drop(index=list(bad_indices))
    .reset_index(drop=True)
)

print(
    "Original validation:",
    len(m2_dataset["validation"])
)

print(
    "Clean validation:",
    len(m2_validation_clean)
)

print(
    "Removed:",
    len(bad_indices)
)

Original validation: 133
Clean validation: 128
Removed: 5


In [ ]:
# ============================================================
# CELL 29: Save excluded validation segments
# ============================================================

excluded_rows = validation_df.iloc[
    sorted(bad_indices)
].copy()

excluded_rows["exclusion_reason"] = (
    "audio-transcript pair is CTC-infeasible; "
    "manual inspection required"
)

EXCLUDED_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "validation_ctc_excluded.csv"
)

excluded_rows.to_csv(
    EXCLUDED_CSV,
    index=False,
    encoding="utf-8"
)

print(excluded_rows[
    [
        "segment_id",
        "speaker_group_id",
        "duration_seconds",
        "exclusion_reason"
    ]
])

         segment_id speaker_group_id  duration_seconds  \
111  REC138_SEG0025           SPK010               2.0   
118  REC138_SEG0033           SPK010               6.0   
126  REC138_SEG0041           SPK010               4.0   
127  REC138_SEG0042           SPK010               2.0   
130  REC138_SEG0045           SPK010               1.0   

                                      exclusion_reason  
111  audio-transcript pair is CTC-infeasible; manua...  
118  audio-transcript pair is CTC-infeasible; manua...  
126  audio-transcript pair is CTC-infeasible; manua...  
127  audio-transcript pair is CTC-infeasible; manua...  
130  audio-transcript pair is CTC-infeasible; manua...  


In [ ]:
# ============================================================
# CELL 29B: Load original MMS-Tarifit zero-shot model
# ============================================================

import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

ZERO_SHOT_MODEL = "iukocha/mms-tachebdant-from-tarifit"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

processor_zero_shot = AutoProcessor.from_pretrained(
    ZERO_SHOT_MODEL
)

zero_shot_model = Wav2Vec2ForCTC.from_pretrained(
    ZERO_SHOT_MODEL
)

zero_shot_model.to(DEVICE)
zero_shot_model.eval()

print("Model:", ZERO_SHOT_MODEL)
print("Device:", DEVICE)
print("Vocabulary:", len(processor_zero_shot.tokenizer))

processor_config.json:   0%|          | 0.00/299 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Model: iukocha/mms-tachebdant-from-tarifit
Device: cuda
Vocabulary: 46


In [ ]:
# ============================================================
# CELL 29C: Define MMS normalization
# ============================================================

import re

def normalize_mms_output(text):
    text = text.lower()

    # MMS orthography -> Corpus V1.1 orthography
    text = text.replace("ə", "e")
    text = text.replace("š", "c")

    # Remove punctuation
    text = re.sub(r"[!,.?\-]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Test
test = "di lməɣrim, nəššin mamšira niɛišə."

print("RAW:       ", test)
print("NORMALIZED:", normalize_mms_output(test))

RAW:        di lməɣrim, nəššin mamšira niɛišə.
NORMALIZED: di lmeɣrim neccin mamcira niɛice


In [ ]:
# ============================================================
# CELL 30: Re-evaluate zero-shot MMS on clean validation set
# ============================================================

# Build clean validation metadata
validation_clean_df = validation_df_clean.copy()

# Reuse the zero-shot MMS model/processor notebook logic
# If zero-shot model is not loaded in this runtime,
# reload iukocha/mms-tachebdant-from-tarifit first.

clean_results = []

for _, row in validation_clean_df.iterrows():

    audio_file = PROJECT_ROOT / row["audio_path"]

    audio, sampling_rate = sf.read(audio_file)

    inputs = processor_zero_shot(
        audio,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )

    input_values = inputs["input_values"].to(DEVICE)

    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    with torch.no_grad():
        outputs = zero_shot_model(
            input_values=input_values,
            attention_mask=attention_mask
        )

    predicted_ids = torch.argmax(
        outputs.logits,
        dim=-1
    )

    prediction_raw = processor_zero_shot.batch_decode(
        predicted_ids
    )[0]

    prediction_normalized = normalize_mms_output(
        prediction_raw
    )

    clean_results.append({
        "segment_id": row["segment_id"],
        "reference": row["transcription"],
        "prediction_raw": prediction_raw,
        "prediction_normalized": prediction_normalized,
    })

print("Finished:", len(clean_results))

Finished: 128


In [ ]:
# ============================================================
# CELL 31: Clean zero-shot MMS metrics
# ============================================================

from jiwer import wer, cer

refs = [
    x["reference"]
    for x in clean_results
]

raw_hyps = [
    x["prediction_raw"]
    for x in clean_results
]

norm_hyps = [
    x["prediction_normalized"]
    for x in clean_results
]

print("CLEAN VALIDATION — RAW MMS")
print(
    "WER:",
    round(wer(refs, raw_hyps) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(refs, raw_hyps) * 100, 2),
    "%"
)

print("\nCLEAN VALIDATION — NORMALIZED MMS")
print(
    "WER:",
    round(wer(refs, norm_hyps) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(refs, norm_hyps) * 100, 2),
    "%"
)

CLEAN VALIDATION — RAW MMS
WER: 93.07 %
CER: 49.25 %

CLEAN VALIDATION — NORMALIZED MMS
WER: 78.88 %
CER: 41.0 %
